# Calenda Labeler — 로컬 Qwen으로 has_schedule 3-way 라벨 (Haiku 대체)

Kaggle **T4×2** 권장. `schedule_criterion.md`를 기준으로 입력 JSONL을 **yes/pending/no**로 분류.
1.x 모델 로드 → 2 라벨링 → 3 Haiku 정답 일치율 검증.
- 기본 모델 `Qwen/Qwen3-14B`. 더 강하게: `Qwen/Qwen3-32B`(T4×2 4bit) 또는 `Qwen/Qwen3.6-27B`.
- 입력: 라벨할 JSONL(레코드: channel·received_at·sender·message·[thread_context]). 출력: `has_schedule` 채운 JSONL.

## 1.1 repo clone (criterion·_common 사용)

In [ ]:
import os
%cd /kaggle/working
if not os.path.exists('Calenda'):
    !git clone https://github.com/sooryong/Calenda.git
else:
    !cd Calenda && git fetch origin && git reset --hard origin/main
%cd /kaggle/working/Calenda
!git log --oneline -1

## 1.2 설치 (vllm)

In [ ]:
!pip -q install vllm 2>/dev/null
import torch
print('GPU:', torch.cuda.device_count(), 'x', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1.3 설정 + criterion 로드

In [ ]:
# ── 설정 ──
MODEL   = 'Qwen/Qwen3-14B'                 # 더 강하게: 'Qwen/Qwen3-32B'(4bit) / 'Qwen/Qwen3.6-27B'
IN_FILE = 'data/processed/train.jsonl.pre_c1bak'   # 라벨할 입력(원본 2133)
OUT_FILE= '/kaggle/working/labeled.jsonl'
QUANT   = None                              # 32B면 'bitsandbytes' 또는 AWQ 모델명 사용

from pathlib import Path
CRITERION = Path('prompts/schedule_criterion.md').read_text(encoding='utf-8')
SYSTEM = ('너는 \'일정 분류기\'다. 아래 [기준]을 유일한 절대 기준으로, 메시지의 has_schedule을 '
          'yes/pending/no 중 하나로 분류한다. JSON 하나만 출력: '
          '{"has_schedule": "yes"|"pending"|"no", "reason": "한 줄"}\n'
          'yes=확정 일정(회의·예약·면접·확정약속). pending=미래 행사/행동 가능성'
          '(공고·모집·마감·설명회·포럼·교육·출범식·초대·미확정 제안). '
          'no=비-일정(거래·결제·적립·배송·인증·광고·인사·과거·남의일정). '
          '날짜·시각 있어도 거래/통보면 no. 거래 안내=no, 행사/모집 안내=pending.\n\n[기준]\n' + CRITERION)
import sys; sys.path.insert(0, 'scripts')
from _common import build_user_block
print('criterion', len(CRITERION), 'chars | 입력', sum(1 for _ in open(IN_FILE, encoding='utf-8')), '건')

## 1.4 모델 로드 (vllm, T4 자동 tensor-parallel)

In [ ]:
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
_tp = max(1, torch.cuda.device_count())
_kw = dict(model=MODEL, tensor_parallel_size=_tp, dtype='bfloat16',
           gpu_memory_utilization=0.92, max_model_len=4096, enforce_eager=True)
if QUANT: _kw['quantization'] = QUANT
llm = LLM(**_kw)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
SP = SamplingParams(temperature=0.0, max_tokens=200, stop=['<|im_end|>'])
print('로드 완료:', MODEL, '| tp=', _tp)

## 2 라벨링 (배치)

In [ ]:
import json, re
def build_prompt(rec):
    ub = build_user_block({'channel':rec.get('channel',''),'received_at':rec.get('received_at',''),
                           'sender':rec.get('sender',''),'message':rec.get('message',''),
                           'thread_context':rec.get('thread_context')})
    msgs=[{'role':'system','content':SYSTEM},{'role':'user','content':ub}]
    # Qwen3 thinking 끔 → 바로 JSON
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)

rows=[json.loads(l) for l in open(IN_FILE,encoding='utf-8') if l.strip()]
prompts=[build_prompt(r) for r in rows]
outs=llm.generate(prompts, SP)
def parse(t):
    m=re.search(r'\{.*\}', t, re.S)
    if not m: return None,'noJSON'
    try:
        o=json.loads(m.group(0)); lab=(o.get('has_schedule') or '').strip().lower()
        return (lab if lab in ('yes','pending','no') else None), (o.get('reason') or '')[:80]
    except Exception: return None,'parseErr'
from collections import Counter
dist=Counter(); n_err=0
for r,o in zip(rows,outs):
    lab,rs=parse(o.outputs[0].text)
    if lab is None: n_err+=1; continue
    r['_label_reason']=rs
    g=r.setdefault('gold',{}); had=bool(g.get('events'))
    if lab=='no': g['has_schedule']='no'; g['events']=[]
    elif had: g['has_schedule']=lab; g['events']=[{k:v for k,v in e.items() if k!='confidence'} for e in g['events']]
    else: g['has_schedule']='no'; g['events']=[]; r['_audit_flag']=lab+'(needs events)'
    dist[lab]+=1
open(OUT_FILE,'w',encoding='utf-8').write('\n'.join(json.dumps(r,ensure_ascii=False) for r in rows)+'\n')
print('분포:', dict(dist), '| 파싱오류', n_err, '→', OUT_FILE)

## 3 ★ Haiku 정답과 일치율 검증 (≥90%면 teacher 채택)

In [ ]:
# Haiku 3-way 라벨(_audited3.jsonl)이 repo에 있으면 같은 메시지로 일치율 측정.
import json
HAIKU='data/processed/_audited3.jsonl'
import os
if os.path.exists(HAIKU):
    hk={ (r.get('channel',''),r.get('message','')): r['gold'].get('has_schedule')
         for r in (json.loads(l) for l in open(HAIKU,encoding='utf-8') if l.strip()) }
    me={ (r.get('channel',''),r.get('message','')): r['gold'].get('has_schedule')
         for r in (json.loads(l) for l in open(OUT_FILE,encoding='utf-8') if l.strip()) }
    keys=[k for k in me if k in hk and hk[k] in ('yes','pending','no')]
    agree=sum(1 for k in keys if me[k]==hk[k])
    from collections import Counter
    conf=Counter((hk[k],me[k]) for k in keys if hk[k]!=me[k])
    print(f'겹침 {len(keys)} | 일치 {agree} = {agree/max(1,len(keys))*100:.1f}%')
    print('주요 불일치(Haiku→Qwen):', conf.most_common(6))
    print('→ 90%+ 면 이 모델을 teacher로 채택. 미만이면 더 큰 모델/Haiku 한도상향.')
else:
    print('_audited3.jsonl 없음 — 검증 스킵(이 입력엔 Haiku 정답 없음).')

## 4 다운로드
`/kaggle/working/labeled.jsonl`을 Output 패널에서 받아 repo로.